# Lev: architecture tests (experiments 1, 2, 2b, 3b, 4 and 6)
Notebook settings: **Accelerator = GPU T4 x2** (the notebook uses only one GPU, as for Laya), **Internet = On**.
First run with `QUICK = True`. If everything passes, set it to `False` and use *Save Version → Save & Run All*.
Same texts, options and seeds as the Laya notebook, so that the comparison is paired.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U transformers "lev[train] @ git+https://github.com/InterfazeAI/lev@0ea955f5aee0d56d906121d1779af20fa8b922ed#subdirectory=packages/lev" umap-learn

In [ ]:
# 0. Configuration
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # a single T4, as for Laya; must be set before importing torch

import json, time, re, sys, platform, logging, zipfile
from contextlib import contextmanager
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

QUICK = True
DTYPE = "float16"   # the T4 has no native bf16; if the sanity check fails (NaN), try "bfloat16"

CFG = dict(
    seed=42,
    exp1_n_options=[2, 10, 50, 150, 500], exp1_n_examples=100, exp1_latency_repeats=30,
    exp2_n_options=[4, 10, 20, 50, 77],   exp2_n_examples=100, exp2_n_perms=5,
    exp2b_n_examples=100, exp4_per_intent=20, exp6_per_intent=25,
)
if QUICK:
    CFG.update(exp1_n_examples=5, exp1_latency_repeats=5, exp2_n_examples=5, exp2_n_perms=3,
               exp2b_n_examples=5, exp4_per_intent=2, exp6_per_intent=4)
RUN = dict(exp1=True, exp2=True, exp2b=True, exp3b=True, exp4=True, exp6=True)

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = ROOT / "lev_bench"
for d in ["results", "logs", "figures"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d-%H%M%S") + ("-quick" if QUICK else "")
SFX = "_quick" if QUICK else ""

log = logging.getLogger("lev_bench"); log.setLevel(logging.INFO); log.handlers.clear()
for h in [logging.FileHandler(OUT / "logs" / f"run_{RUN_ID}.log"), logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")); log.addHandler(h)
log.info(f"run_id={RUN_ID} out={OUT}")

In [ ]:
# 1. Model, metadata and sanity check
# Same procedure as lev.load(), with one difference: the precision is configurable
# (lev.load hard-codes bfloat16, which the T4 does not run natively).
import transformers, peft, datasets, lev
from transformers import AutoModelForCausalLM, AutoTokenizer
from lev.model import (DecisionEngine, EngineConfig, _load_head, serving_routes, _QUESTIONS)
from lev.calibrate import CalibrationProfile
from lev.train.checkpoints import resolve_checkpoint, CALIBRATION, MODE_B_HEAD
from lev.prompt import build, label_prefix, Style, Layout
from lev.readout.mode_a import LabelTokenReadout
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.perf_counter()
ckpt = resolve_checkpoint("interfaze-ai/lev")
manifest = json.loads((ckpt / "lev_release.json").read_text())
TOK = AutoTokenizer.from_pretrained(str(ckpt if (ckpt / "tokenizer.json").is_file() else manifest["base_model"]))
base = AutoModelForCausalLM.from_pretrained(manifest["base_model"], dtype=getattr(torch, DTYPE)).to(DEVICE)
MODEL = PeftModel.from_pretrained(base, str(ckpt)).eval()
head = _load_head(ckpt / MODE_B_HEAD, MODEL)
engine = DecisionEngine(
    MODEL, TOK,
    EngineConfig(model_id=manifest["base_model"], noul_readout="rating",
                 prompt_style=manifest.get("prompt_style", "plain"), max_label_options=None,
                 skip_multi_token_codes=True),
    CalibrationProfile.load(ckpt / CALIBRATION) if (ckpt / CALIBRATION).is_file() else CalibrationProfile(),
    mode_b_head=head)
STYLE = Style(engine.config.prompt_style)

def has(mod):
    try: __import__(mod); return True
    except Exception: return False

meta = dict(run_id=RUN_ID, quick=QUICK, cfg=CFG, python=sys.version.split()[0], platform=platform.platform(),
            torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,
            datasets=datasets.__version__, lev=lev.__version__, lev_commit="0ea955f5",
            manifest=manifest, dtype=DTYPE, device=DEVICE, gpus_visible=torch.cuda.device_count(),
            gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
            mode_b_head=head is not None, prompt_style=STYLE.value,
            fast_kernels=dict(fla=has("fla"), causal_conv1d=has("causal_conv1d")),
            load_seconds=round(time.perf_counter() - t0, 1))

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def timed(state, questions):
    sync(); t = time.perf_counter()
    r = engine.system_one(state, questions)
    sync(); return r, (time.perf_counter() - t) * 1000

sanity_q = {"intent": {"type": "choice", "instructions": "What does the customer want?",
            "criteria": {"refund": "wants money back", "cancel": "wants to cancel an order",
                         "track": "wants to know where an order is", "other": "anything else"}}}
sanity_state = "Hi, I was charged twice for my order #4471 and I want a refund."
for _ in range(3): engine.system_one(sanity_state, sanity_q)
lat = [timed(sanity_state, sanity_q)[1] for _ in range(20)]
a = engine.system_one(sanity_state, sanity_q).answers["intent"]
meta["sanity"] = dict(choice=a.choice, probabilities=a.probabilities, latency_ms_median=float(np.median(lat)),
                      readme_reference={"refund": 0.84, "cancel": 0.094, "track": 0.012, "other": 0.054})
(OUT / "results" / f"run_meta_{RUN_ID}.json").write_text(json.dumps(meta, indent=2, default=str))
log.info(f"meta: {json.dumps({k: meta[k] for k in ['gpu', 'dtype', 'prompt_style', 'mode_b_head', 'fast_kernels', 'transformers']})}")
log.info(f"sanity: {meta['sanity']}")
assert not any(np.isnan(v) for v in a.probabilities.values()), "NaN in the probabilities: change DTYPE"
assert a.choice == "refund", "Sanity check failed: the README example did not return 'refund'"

In [ ]:
# 2. Utilities: resumable JSONL logs, Lev regimes, route of each question
def jdefault(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return str(o)

class JsonlLog:
    """One line per prediction. Keys already written are skipped (resume within a session)."""
    def __init__(self, name):
        self.path = OUT / "results" / f"{name}{SFX}.jsonl"
        self.done = set()
        if self.path.exists():
            for line in self.path.open():
                try: self.done.add(json.loads(line)["key"])
                except Exception: pass
    def write(self, rec):
        rec = {"run_id": RUN_ID, **rec}
        with self.path.open("a") as f: f.write(json.dumps(rec, default=jdefault) + "\n")
        self.done.add(rec["key"])
    def df(self):
        return pd.read_json(self.path, lines=True) if self.path.exists() else pd.DataFrame()

def humanize(label):
    return re.sub(r"^in:", "", str(label).strip().lower()).replace("_", " ").strip()

def choice_q(criteria, instructions):
    return {"q": {"type": "choice", "instructions": instructions, "criteria": criteria}}

def tv(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in keys)

def savefig(name):
    plt.tight_layout(); plt.savefig(OUT / "figures" / f"{name}{SFX}.png", dpi=160); plt.show()

# Lev regimes:
#   A        = as shipped: single-token codes read at the answer position, averaged over 2 orders
#   A_noavg  = the same mechanism without the 2-order average (the raw mechanism)
#   B        = forces the alternative path: the head that compares the state with the text of each option
REGIMES = {"A": dict(max_label_options=None, order_average=True),
           "A_noavg": dict(max_label_options=None, order_average=False),
           "B": dict(max_label_options=1, order_average=True)}
if engine.mode_b_head is None:
    REGIMES.pop("B"); log.info("no mode B head in the checkpoint: regime B removed")

@contextmanager
def regime(name):
    old = (engine.config.max_label_options, engine.config.order_average)
    engine.config.max_label_options = REGIMES[name]["max_label_options"]
    engine.config.order_average = REGIMES[name]["order_average"]
    try: yield
    finally: engine.config.max_label_options, engine.config.order_average = old

def route_info(qs):
    rt = serving_routes(_QUESTIONS.validate_python(qs), TOK, engine.config)["q"]
    return dict(mode=rt.mode.value, n_codes=len(rt.codes or []), route_reason=rt.reason,
                max_code_len=max(len(c) for c in rt.codes) if rt.codes else None)

def ask(state, qs):
    """One question, with its latency and Lev's usage record."""
    r, ms = timed(state, qs)
    a = r.answers["q"]
    return a.choice, dict(a.probabilities), ms, dict(input_tokens=r.usage.input_tokens,
                                                    cached_input_tokens=r.usage.cached_input_tokens)

In [ ]:
# 3. Data: same label pool as Laya (banking77 + clinc150 + MASSIVE = 284)
from datasets import load_dataset
b77 = load_dataset("mteb/banking77", split="test").to_pandas()[["text", "label_text"]]
B77_LABELS = sorted(b77["label_text"].unique())
POOL, seen, POOL_SOURCES = [], set(), {}
def add_to_pool(labels, source):
    n0 = len(POOL)
    for lab in labels:
        h = humanize(lab)
        if h and h not in seen and h != "oos":
            seen.add(h); POOL.append((h, source))
    POOL_SOURCES[source] = len(POOL) - n0
add_to_pool(B77_LABELS, "banking77")
add_to_pool(load_dataset("clinc/clinc_oos", "plus", split="test").features["intent"].names, "clinc150")
ds = load_dataset("mteb/amazon_massive_intent", "en", split="test")
add_to_pool(sorted(set(ds["label_text" if "label_text" in ds.column_names else "label"])), "mteb/amazon_massive_intent")
log.info(f"label pool: {len(POOL)} unique, by source: {POOL_SOURCES} (Laya used 284)")
json.dump({"pool_size": len(POOL), "by_source": POOL_SOURCES, "labels": POOL},
          open(OUT / "results" / "label_pool.json", "w"), indent=1)

INSTR = "What does the customer want? Pick the intent that matches the message."

def option_set(gold_h, n, rng):
    b = [h for h, s in POOL if s == "banking77" and h != gold_h]
    rest = [h for h, s in POOL if s != "banking77"]
    rng.shuffle(b); rng.shuffle(rest)
    opts = [gold_h] + (b + rest)[: n - 1]
    rng.shuffle(opts)
    return opts

def sample_b77(n, seed):
    return b77.sample(n=min(n, len(b77)), random_state=seed).reset_index(drop=True)

In [ ]:
# 4. Experiment 1: scaling with the number of options (regimes A and B)
# A: every option enters the text that is read, in 2 orders -> latency should grow with N.
# B: the options stay out of the prompt; each becomes a vector, cached per option set.
#    In the latency benchmark (same set repeated) the cache is warm, as in CLM.
#    In the accuracy run each text has a new set, so the cache is cold.
if RUN["exp1"]:
    L1, LB = JsonlLog("exp1_scaling"), JsonlLog("exp1_latency_bench")
    ex = sample_b77(CFG["exp1_n_examples"], CFG["seed"])
    for reg in [r for r in ["A", "B"] if r in REGIMES]:
        with regime(reg):
            for n_req in CFG["exp1_n_options"]:
                n = min(n_req, len(POOL))
                rng = np.random.default_rng(CFG["seed"] + n)
                s0 = ex.loc[0, "text"]; qs0 = choice_q({o: None for o in option_set(humanize(ex.loc[0, "label_text"]), n, rng)}, INSTR)
                engine._candidate_cache.clear()
                try:
                    for _ in range(3): engine.system_one(s0, qs0)
                    for r in range(CFG["exp1_latency_repeats"]):
                        k = f"{reg}|{n_req}|{r}"
                        if k in LB.done: continue
                        _, _, ms, use = ask(s0, qs0)
                        LB.write(dict(key=k, regime=reg, n_req=n_req, n=n, rep=r, latency_ms=ms, cache="warm" if reg == "B" else "n/a",
                                      **use, **route_info(qs0), status="ok"))
                except Exception as e:
                    LB.write(dict(key=f"{reg}|{n_req}|error", regime=reg, n_req=n_req, n=n, rep=-1,
                                  status=type(e).__name__, error=str(e)[:200]))
                    if DEVICE == "cuda": torch.cuda.empty_cache()
                for i, row in ex.iterrows():
                    k = f"{reg}|{n_req}|{i}"
                    if k in L1.done: continue
                    gold = humanize(row["label_text"])
                    rng = np.random.default_rng(CFG["seed"] * 1000 + i * 7 + n)
                    qs = choice_q({o: None for o in option_set(gold, n, rng)}, INSTR)
                    engine._candidate_cache.clear()  # mode B: cold cache; also bounds memory use
                    rec = dict(key=k, regime=reg, n_req=n_req, n=n, idx=i, gold=gold, **route_info(qs))
                    try:
                        pred, p, ms, use = ask(row["text"], qs)
                        rec.update(pred=pred, correct=pred == gold, p_gold=p.get(gold, 0.0), latency_ms=ms,
                                   cache="cold" if reg == "B" else "n/a", **use,
                                   top5=dict(sorted(p.items(), key=lambda kv: -kv[1])[:5]), status="ok")
                    except Exception as e:
                        rec.update(status=type(e).__name__, error=str(e)[:200])
                        if DEVICE == "cuda": torch.cuda.empty_cache()
                    L1.write(rec)
                log.info(f"exp1 {reg} N={n_req} ok")
    engine._candidate_cache.clear()

    d1, db = L1.df(), LB.df()
    ok = d1[d1.status == "ok"]
    s1 = d1.groupby(["regime", "n_req"]).agg(n_eff=("n", "first"), mode=("mode", "first"), n_codes=("n_codes", "mean"),
                                             fail_rate=("status", lambda s: (s != "ok").mean())).join(
        ok.groupby(["regime", "n_req"]).agg(accuracy=("correct", "mean"), p_gold=("p_gold", "mean"),
                                            input_tokens=("input_tokens", "mean"), lat_cold_median=("latency_ms", "median")))
    dbo = db[db.status == "ok"]
    s1 = s1.join(dbo.groupby(["regime", "n_req"]).latency_ms.agg(lat_median="median", lat_p90=lambda x: np.percentile(x, 90)))
    s1.to_csv(OUT / "results" / f"exp1_summary{SFX}.csv"); display(s1)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for reg, g in s1.reset_index().groupby("regime"):
        ax[0].plot(g.n_req, g.lat_median, "o-", label=f"{reg} (repeated set)")
        if reg == "B": ax[0].plot(g.n_req, g.lat_cold_median, "o--", label="B (new set)")
        ax[1].plot(g.n_req, g.accuracy, "o-", label=reg)
        ax[2].plot(g.n_req, g.input_tokens, "o-", label=reg)
    for a_, t in zip(ax, ["median latency (ms)", "accuracy", "tokens read per question"]):
        a_.set_xscale("log"); a_.set_xlabel("number of options"); a_.set_title(t); a_.legend()
    savefig("exp1_scaling")

In [ ]:
# 5. Experiment 2: invariance to option order (A, A_noavg, B)
# Predictions: B never changes (options become vectors without position); A_noavg changes; A changes less than A_noavg.
if RUN["exp2"]:
    L2 = JsonlLog("exp2_order")
    ex = sample_b77(CFG["exp2_n_examples"], CFG["seed"] + 1)
    for reg in REGIMES:
        with regime(reg):
            for n in CFG["exp2_n_options"]:
                for i, row in ex.iterrows():
                    gold = humanize(row["label_text"])
                    rng = np.random.default_rng(CFG["seed"] * 2000 + i * 11 + n)
                    base_opts = option_set(gold, n, rng)
                    for pi in range(CFG["exp2_n_perms"]):
                        k = f"{reg}|{n}|{i}|{pi}"
                        order = list(base_opts) if pi == 0 else list(rng.permutation(base_opts))
                        if k in L2.done: continue
                        qs = choice_q({o: None for o in order}, INSTR)
                        engine._candidate_cache.clear()
                        rec = dict(key=k, regime=reg, n=n, idx=i, perm=pi, gold=gold, gold_pos=order.index(gold))
                        try:
                            pred, p, ms, use = ask(row["text"], qs)
                            rec.update(pred=pred, correct=pred == gold, probs=p, p_gold=p.get(gold, 0.0), status="ok")
                        except Exception as e:
                            rec.update(status=type(e).__name__, error=str(e)[:200])
                        L2.write(rec)
                log.info(f"exp2 {reg} N={n} ok")

    d2 = L2.df(); d2 = d2[d2.status == "ok"]
    rows = []
    for (reg, n, i), g in d2.groupby(["regime", "n", "idx"]):
        P = list(g.probs)
        rows.append(dict(regime=reg, n=n, idx=i, flipped=g.pred.nunique() > 1,
                         mean_tv=np.mean([tv(P[a], P[b]) for a in range(len(P)) for b in range(a + 1, len(P))]),
                         p_gold_std=g.p_gold.std(), accuracy=g.correct.mean()))
    s2 = pd.DataFrame(rows).groupby(["regime", "n"]).agg(flip_rate=("flipped", "mean"), mean_tv=("mean_tv", "mean"),
                                                        p_gold_std=("p_gold_std", "mean"), accuracy=("accuracy", "mean"))
    d2["pos_bin"] = pd.cut(d2.gold_pos / (d2.n - 1).clip(lower=1), [-0.01, 0.25, 0.5, 0.75, 1.0],
                           labels=["first", "second", "third", "last"])
    pos = d2.groupby(["regime", "n", "pos_bin"], observed=True).correct.mean().unstack()
    s2.to_csv(OUT / "results" / f"exp2_summary{SFX}.csv"); pos.to_csv(OUT / "results" / f"exp2_position_bias{SFX}.csv")
    display(s2); display(pos)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for reg, g in s2.reset_index().groupby("regime"):
        ax[0].plot(g.n, g.flip_rate, "o-", label=reg); ax[1].plot(g.n, g.mean_tv, "o-", label=reg)
    for a_, t in zip(ax, ["fraction of texts whose answer changes", "mean distance between distributions (TV)"]):
        a_.set_xlabel("number of options"); a_.set_title(t); a_.legend()
    savefig("exp2_order")

In [ ]:
# 6. Experiment 2b: position in the list or distance in tokens? (same design as Laya)
# In Lev the text comes BEFORE the options and the answer is read at the end of the prompt. The distance measured here
# is from the correct option's line to the answer position. Runs in regime A_noavg: with the 2-order average,
# each option would appear in two positions and the design would no longer hold.
if RUN["exp2b"]:
    import statsmodels.formula.api as smf
    N2B, POS2B = 50, [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 49]
    LAYOUTS = {"short": None, "long": "banking intent"}
    L2b = JsonlLog("exp2b_distance")
    ex = sample_b77(CFG["exp2b_n_examples"], CFG["seed"] + 21)
    with regime("A_noavg"):
        for i, row in ex.iterrows():
            gold = humanize(row["label_text"])
            rng = np.random.default_rng(CFG["seed"] * 5000 + i)
            distract = [o for o in option_set(gold, N2B, rng) if o != gold]
            for layout, desc in LAYOUTS.items():
                for p in POS2B:
                    k = f"{layout}|{p}|{i}"
                    if k in L2b.done: continue
                    order = distract[:p] + [gold] + distract[p:]
                    qs = choice_q({o: desc for o in order}, INSTR)
                    qv = _QUESTIONS.validate_python(qs)
                    codes = serving_routes(qv, TOK, engine.config)["q"].codes
                    suffix = build(row["text"], "q", qv["q"], codes, Layout.STATE_FIRST, None, None, STYLE).suffix
                    j = suffix.find(f"\n{codes[p]}. {gold}" if STYLE is Style.CHAT else f"\n  {codes[p]}: {gold}")
                    dist = len(TOK.encode(suffix[j:], add_special_tokens=False)) if j >= 0 else None
                    pred, probs, ms, use = ask(row["text"], qs)
                    L2b.write(dict(key=k, layout=layout, pos=p, idx=i, gold=gold, pred=pred, correct=pred == gold,
                                   p_gold=probs[gold], rank_gold=sorted(probs.values(), reverse=True).index(probs[gold]) + 1,
                                   dist_tokens=dist, input_tokens=use["input_tokens"], code=codes[p]))
            if i % 20 == 0: log.info(f"exp2b: {i + 1}/{len(ex)} texts")

    d = L2b.df()
    s_pos = d.groupby(["layout", "pos"]).agg(dist_tokens=("dist_tokens", "mean"), accuracy=("correct", "mean"),
                                             p_gold=("p_gold", "mean"), rank_gold=("rank_gold", "mean"),
                                             input_tokens=("input_tokens", "mean"))
    d["y"] = d.correct.astype(int); d["pos_std"] = d.pos / 49; d["dist_std"] = d.dist_tokens / 100
    try:
        fit = smf.logit("y ~ pos_std + dist_std", data=d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": d.idx})
        coefs = pd.DataFrame({"coef": fit.params, "p_value": fit.pvalues})
    except Exception as e:
        log.info(f"exp2b: logistic regression failed ({e!r})"); coefs = pd.DataFrame()
    s_pos.to_csv(OUT / "results" / f"exp2b_by_position{SFX}.csv"); coefs.to_csv(OUT / "results" / f"exp2b_logit{SFX}.csv")
    display(s_pos); display(coefs)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for layout, g in s_pos.reset_index().groupby("layout"):
        ax[0].plot(g.pos, g.accuracy, "o-", label=layout); ax[1].plot(g.dist_tokens, g.accuracy, "o-", label=layout)
    ax[0].set_xlabel("position of the correct option in the list (0 = first)")
    ax[1].set_xlabel("distance in tokens from the correct option to the answer position")
    for a_ in ax: a_.set_ylabel("accuracy"); a_.set_ylim(0, 1); a_.legend()
    savefig("exp2b_distance")

In [ ]:
# 7. Experiment 3b: minimal-edit pairs (same set as Laya), regime A
T3B = [
    ("count", "change", "I was charged once for my {0}.", "I was charged twice for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("A", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("count", "change", "I entered the wrong PIN once {0}.", "I entered the wrong PIN three times {0}.",
     ("pin question", "a general question about the PIN"), ("card blocked", "the card is blocked after too many wrong PIN attempts"),
     ("A", "B"), [("at the ATM",), ("at a shop",), ("at the petrol station",)]),
    ("count", "change", "I paid for one {0} and received one.", "I paid for two {1} and received one.",
     ("order complete", "everything that was paid for was delivered"), ("missing item", "fewer items were delivered than were paid for"),
     ("A", "B"), [("ticket", "tickets"), ("shirt", "shirts"), ("book", "books")]),
    ("count", "change", "My {0} was paid into my account once this month.", "My {0} was paid into my account twice this month.",
     ("normal payment", "an expected single payment was received"), ("duplicate payment", "the same payment was received more than once"),
     ("A", "B"), [("salary",), ("pension",), ("allowance",)]),
    ("negation", "change", "I want to cancel my {0}.", "I do not want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("A", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("negation", "change", "The {0} arrived.", "The {0} never arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("A", "B"), [("package",), ("card",), ("order",)]),
    ("negation", "change", "I recognise this {0} on my statement.", "I don't recognise this {0} on my statement.",
     ("confirmed", "the customer confirms it is theirs"), ("fraud report", "the customer did not make it"),
     ("A", "B"), [("payment",), ("transfer",), ("charge",)]),
    ("negation", "change", "The {0} works after the update.", "The {0} no longer works after the update.",
     ("working fine", "it works as expected"), ("bug", "it is broken or not working"),
     ("A", "B"), [("app",), ("website",), ("card reader",)]),
    ("numbers", "change", "My balance is ${0} and I want to withdraw ${1}.", "My balance is ${1} and I want to withdraw ${0}.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("A", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("numbers", "change", "The invoice says ${0} and I was charged ${0}.", "The invoice says ${0} and I was charged ${1}.",
     ("amount correct", "charged exactly the invoice amount"), ("amount mismatch", "charged a different amount than the invoice"),
     ("A", "B"), [("120", "210"), ("45", "54"), ("310", "130")]),
    ("numbers", "change", "I ordered size {0} and received size {0}.", "I ordered size {0} and received size {1}.",
     ("correct item", "received exactly the size ordered"), ("wrong item", "received a different size than ordered"),
     ("A", "B"), [("42", "24"), ("38", "83"), ("46", "64")]),
    ("numbers", "change", "My flight leaves at {0} and I will get to the airport at {1}.", "My flight leaves at {1} and I will get to the airport at {0}.",
     ("on time", "the customer will make the flight"), ("missed flight", "the customer will miss the flight"),
     ("A", "B"), [("9", "7"), ("11", "8"), ("6", "4")]),
    ("roles", "change", "{1} sent money to me.", "I sent money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("A", "B"), [("the bank", "The bank"), ("my landlord", "My landlord"), ("my friend", "My friend")]),
    ("roles", "change", "I refunded the {0}.", "The {0} refunded me.",
     ("refund issued", "the writer gave money back"), ("refund received", "the writer got money back"),
     ("A", "B"), [("customer",), ("shop",), ("seller",)]),
    ("roles", "change", "I owe {0} money.", "{1} owes me money.",
     ("writer owes", "the writer has a debt to someone"), ("owed to writer", "someone has a debt to the writer"),
     ("A", "B"), [("my brother", "My brother"), ("the shop", "The shop"), ("my landlord", "My landlord")]),
    ("roles", "change", "{1} paid me.", "I paid {0}.",
     ("money received", "the writer received a payment"), ("money paid", "the writer made a payment"),
     ("A", "B"), [("my employer", "My employer"), ("my tenant", "My tenant"), ("my client", "My client")]),
    ("time", "change", "I want to pay before {0}.", "I want to pay after {0}.",
     ("on-time payment", "paying before the deadline"), ("late payment", "paying after the deadline"),
     ("A", "B"), [("the due date",), ("the deadline",), ("the end of the month",)]),
    ("time", "change", "I will travel {0} next week.", "I travelled {0} last week.",
     ("future trip", "about a trip that has not happened yet"), ("past trip", "about a trip that already happened"),
     ("A", "B"), [("abroad",), ("to Spain",), ("to Japan",)]),
    ("time", "change", "I want to stop a {0} that will go out tomorrow.", "I want to stop a {0} that went out yesterday.",
     ("stop future payment", "cancel a payment before it is sent"), ("reverse past payment", "get back a payment that was already sent"),
     ("A", "B"), [("payment",), ("transfer",), ("direct debit",)]),
    ("time", "change", "My {0} renews next month.", "My {0} renewed last month.",
     ("upcoming renewal", "a renewal that has not happened yet"), ("past renewal", "a renewal that already happened"),
     ("A", "B"), [("subscription",), ("membership",), ("insurance",)]),
    ("quantifier", "change", "All of my {0} failed today.", "None of my {0} failed today.",
     ("failed", "they did not go through"), ("went through", "they went through normally"),
     ("A", "B"), [("transfers",), ("payments",), ("top-ups",)]),
    ("quantifier", "change", "Every {0} I tried was declined.", "No {0} I tried was declined.",
     ("declined", "the attempts were rejected"), ("accepted", "the attempts were accepted"),
     ("A", "B"), [("card payment",), ("withdrawal",), ("transfer",)]),
    ("quantifier", "change", "All of my {0} were refunded.", "None of my {0} were refunded.",
     ("refund complete", "the money was returned"), ("refund missing", "the money was not returned"),
     ("A", "B"), [("orders",), ("purchases",), ("tickets",)]),
    ("quantifier", "change", "I received all of my {0}.", "I received none of my {0}.",
     ("all delivered", "everything arrived"), ("nothing delivered", "nothing arrived"),
     ("A", "B"), [("parcels",), ("cards",), ("documents",)]),
    # controls: the edit does not change the meaning, so the answer should not change
    ("paraphrase", "same", "I was charged twice for my {0}.", "I was charged two times for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("B", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("paraphrase", "same", "I do not want to cancel my {0}.", "I don't want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("B", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("paraphrase", "same", "My balance is ${1} and I want to withdraw ${0}.", "My balance is {1} dollars and I want to withdraw {0} dollars.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("B", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("paraphrase", "same", "The {0} never arrived.", "The {0} has not arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("B", "B"), [("package",), ("card",), ("order",)]),
    ("paraphrase", "same", "I sent money to {0}.", "I transferred money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("B", "B"), [("the bank",), ("my landlord",), ("my friend",)]),
]
OTHER3B, INSTR3B = ("other", "anything else"), "Which option best describes the customer's message?"
if RUN["exp3b"]:
    L3b = JsonlLog("exp3b_minimal_pairs")
    with regime("A"):
        for t_id, (cat, kind, ta_t, tb_t, oA, oB, (ga, gb), fills) in enumerate(T3B):
            opt = {"A": oA, "B": oB}
            for f_id, f in enumerate(fills):
                ta, tb = ta_t.format(*f), tb_t.format(*f)
                for order_name, seq in [("fwd", ["A", "B"]), ("rev", ["B", "A"])]:
                    k = f"{t_id}|{f_id}|{order_name}"
                    if k in L3b.done: continue
                    crit = {opt[s][0]: opt[s][1] for s in seq}; crit[OTHER3B[0]] = OTHER3B[1]
                    qs = choice_q(crit, INSTR3B)
                    pa, PA, _, _ = ask(ta, qs); pb, PB, _, _ = ask(tb, qs)
                    gA, gB = opt[ga][0], opt[gb][0]
                    L3b.write(dict(key=k, template=t_id, filler=f_id, category=cat, kind=kind, order=order_name,
                                   text_a=ta, text_b=tb, gold_a=gA, gold_b=gB, pred_a=pa, pred_b=pb,
                                   correct_a=pa == gA, correct_b=pb == gB, flipped=pa != pb, tv=tv(PA, PB),
                                   p_gold_b_in_a=PA[gB], p_gold_b_in_b=PB[gB], probs_a=PA, probs_b=PB))
    d = L3b.df()
    d["both_correct"] = d.correct_a & d.correct_b
    d["shift_to_gold_b"] = d.p_gold_b_in_b - d.p_gold_b_in_a
    d["moved_right_way"] = d.shift_to_gold_b > 0
    agg = dict(runs=("key", "count"), acc_a=("correct_a", "mean"), acc_b=("correct_b", "mean"),
               both_correct=("both_correct", "mean"), flip_rate=("flipped", "mean"), mean_tv=("tv", "mean"),
               shift_to_gold_b=("shift_to_gold_b", "mean"), moved_right_way=("moved_right_way", "mean"))
    s3b = d.groupby(["kind", "category"]).agg(**agg)
    s3b_t = d.groupby(["kind", "category", "template"]).agg(**agg)
    s3b_t["example_a"] = d.groupby(["kind", "category", "template"]).text_a.first()
    s3b.to_csv(OUT / "results" / f"exp3b_summary{SFX}.csv"); s3b_t.to_csv(OUT / "results" / f"exp3b_by_template{SFX}.csv")
    display(s3b)
    s3b.reset_index().set_index("category")[["acc_a", "acc_b", "both_correct", "flip_rate"]].plot.bar(figsize=(11, 4), rot=0, ylim=(0, 1))
    plt.title("minimal edits: accuracy on each side, on both sides, and flip rate (paraphrase = control)")
    savefig("exp3b_minimal_pairs")

In [ ]:
# 8. Experiment 4: labels without meaning (same 12 intents and conditions as Laya), regime A
INTENTS4 = {
    "activate_my_card": "how to activate a newly received card",
    "age_limit": "minimum age required to open or use an account",
    "atm_support": "which cash machines can be used and how",
    "card_swallowed": "a cash machine kept or retained the card",
    "change_pin": "how to change the card PIN code",
    "exchange_rate": "what currency exchange rate is applied",
    "lost_or_stolen_phone": "the phone with the banking app was lost or stolen",
    "terminate_account": "close or delete the account permanently",
    "top_up_limits": "maximum amount that can be added to the account",
    "transaction_charged_twice": "the same payment appears twice",
    "request_refund": "ask for money back for a purchase",
    "edit_personal_details": "update name, address or other personal information",
}
missing = [k for k in INTENTS4 if k not in B77_LABELS]
if missing: log.info(f"exp4: intents missing from banking77, removed: {missing}")
INTENTS4 = {k: v for k, v in INTENTS4.items() if k in B77_LABELS}
I4 = list(INTENTS4)
rng4 = np.random.default_rng(CFG["seed"] + 4)
CODES = {lab: f"K{c}" for lab, c in zip(I4, rng4.choice(np.arange(10, 100), len(I4), replace=False))}
shift = rng4.integers(1, len(I4))
MISLEAD = {lab: humanize(I4[(j + shift) % len(I4)]) for j, lab in enumerate(I4)}  # no key points to its own intent

def criteria_for(cond, order):
    """Returns {shown_key: description} and {shown_key: true intent of the description}."""
    crit, key2int = {}, {}
    for lab in order:
        key = dict(original=humanize(lab), name_only=humanize(lab), code_desc=CODES[lab],
                   code_only=CODES[lab], misleading=MISLEAD[lab])[cond]
        crit[key] = None if cond in ("name_only", "code_only") else INTENTS4[lab]
        key2int[key] = lab
    return crit, key2int
INSTR4 = "Which option matches the customer's request?"
if RUN["exp4"]:
    L4 = JsonlLog("exp4_labels")
    sub = b77[b77.label_text.isin(I4)].groupby("label_text", group_keys=False).apply(
        lambda g: g.sample(n=min(CFG["exp4_per_intent"], len(g)), random_state=CFG["seed"])).reset_index(drop=True)
    with regime("A"):
        for i, row in sub.iterrows():
            order = list(np.random.default_rng(CFG["seed"] * 4000 + i).permutation(I4))
            for cond in ["original", "name_only", "code_desc", "code_only", "misleading"]:
                k = f"{i}|{cond}"
                if k in L4.done: continue
                crit, key2int = criteria_for(cond, order)
                pred, p, _, _ = ask(row["text"], choice_q(crit, INSTR4))
                L4.write(dict(key=k, idx=i, condition=cond, gold=row["label_text"], pred_key=pred,
                              pred_intent=key2int[pred], correct=key2int[pred] == row["label_text"],
                              followed_key=(cond == "misleading" and pred == humanize(row["label_text"])),
                              p_max=max(p.values())))
    d4 = L4.df()
    s4 = d4.groupby("condition").agg(n=("idx", "count"), accuracy=("correct", "mean"),
                                     followed_key=("followed_key", "mean"), p_max=("p_max", "mean"))
    s4 = s4.reindex(["original", "name_only", "code_desc", "code_only", "misleading"]); s4["chance"] = 1 / len(I4)
    s4.to_csv(OUT / "results" / f"exp4_summary{SFX}.csv"); display(s4)
    plt.figure(figsize=(8, 4))
    plt.bar(s4.index, s4.accuracy, label="correct by description")
    plt.bar(["misleading"], s4.loc[["misleading"], "followed_key"], bottom=s4.loc[["misleading"], "accuracy"],
            color="C3", label="followed the misleading key")
    plt.axhline(1 / len(I4), ls="--", c="k", label="chance"); plt.ylim(0, 1); plt.legend()
    plt.title("what Lev uses to decide: key or description?")
    savefig("exp4_labels")

In [ ]:
# 9. Experiment 6: X-ray at the answer position, with and without LoRA
# For each text: a prompt with the 12 options (one order, no averaging), read twice by the same Qwen:
# with the LoRA adapter on and off. Measures how much probability goes to the option codes,
# whether the correct option wins among the codes, and projects the last hidden state to 2D.
if RUN["exp6"]:
    import umap
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    from sklearn.metrics import silhouette_score

    sub = b77[b77.label_text.isin(I4)].groupby("label_text", group_keys=False).apply(
        lambda g: g.sample(n=min(CFG["exp6_per_intent"], len(g)), random_state=CFG["seed"] + 6)).reset_index(drop=True)
    readout = LabelTokenReadout(TOK, prefix=label_prefix(STYLE))
    rows6, H = [], {"base": [], "lora": []}; agree = []
    with regime("A_noavg"):
        for i, row in sub.iterrows():
            order = list(np.random.default_rng(CFG["seed"] * 6000 + i).permutation(I4))
            crit, key2int = criteria_for("original", order)
            qs = choice_q(crit, INSTR4); qv = _QUESTIONS.validate_python(qs)
            routes = serving_routes(qv, TOK, engine.config)
            prefix, variants = engine._render(row["text"], qv, routes)
            ids = TOK.encode(prefix, add_special_tokens=True) + variants[0].suffix_ids
            code_ids = readout.candidate_ids(routes["q"].codes)
            x = torch.tensor([ids], device=DEVICE)
            gi = order.index(row["label_text"])
            for name in ["lora", "base"]:
                with torch.no_grad():
                    if name == "base":
                        with MODEL.disable_adapter():
                            out = MODEL(input_ids=x, output_hidden_states=True, use_cache=False)
                    else:
                        out = MODEL(input_ids=x, output_hidden_states=True, use_cache=False)
                pv = torch.softmax(out.logits[0, -1].float(), -1)
                pc = pv[code_ids]; mass = float(pc.sum()); pcn = (pc / pc.sum()).cpu().numpy()
                top_id = int(pv.argmax())
                rows6.append(dict(example=i, model=name, gold=row["label_text"], mass_on_codes=mass,
                                  correct=int(pcn.argmax()) == gi, p_gold_within_codes=float(pcn[gi]),
                                  top_token=TOK.decode([top_id]), top_token_is_code=top_id in code_ids))
                H[name].append(out.hidden_states[-1][0, -1].float().cpu().numpy())
                if name == "lora":
                    agree.append(key2int[ask(row["text"], qs)[0]] == order[int(pcn.argmax())])
    log.info(f"exp6: manual readout (with LoRA) agrees with engine.system_one on {np.mean(agree):.1%} of texts")
    r6 = pd.DataFrame(rows6); r6.to_csv(OUT / "results" / f"exp6_readout{SFX}.csv", index=False)
    X = {k: np.stack(v) for k, v in H.items()}; y = sub.label_text.values
    np.savez_compressed(OUT / "results" / f"exp6_hidden{SFX}.npz", **X)

    summ, Z = [], {}
    for name, A in X.items():
        An = A / np.linalg.norm(A, axis=1, keepdims=True)
        g = r6[r6.model == name]
        k = min(5, pd.Series(y).value_counts().min())
        probe = cross_val_score(LogisticRegression(max_iter=3000), An, y, cv=StratifiedKFold(k, shuffle=True, random_state=0)).mean() if k >= 2 else np.nan
        summ.append(dict(model=name, mass_on_codes_mean=g.mass_on_codes.mean(), mass_on_codes_median=g.mass_on_codes.median(),
                         top_token_is_code=g.top_token_is_code.mean(), accuracy_within_codes=g.correct.mean(),
                         p_gold_within_codes=g.p_gold_within_codes.mean(), probe_accuracy_intent=probe,
                         silhouette_by_intent=silhouette_score(An, y, metric="cosine"),
                         most_common_top_token=g.top_token.value_counts().index[0]))
        Z[name] = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=CFG["seed"]).fit_transform(An)
    s6 = pd.DataFrame(summ).set_index("model")
    s6.to_csv(OUT / "results" / f"exp6_summary{SFX}.csv"); display(s6)
    np.savez_compressed(OUT / "results" / f"exp6_umap{SFX}.npz", **Z)

    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    for name, c in [("base", "C7"), ("lora", "C0")]:
        ax[0].hist(r6[r6.model == name].mass_on_codes, bins=np.linspace(0, 1, 21), alpha=.6, color=c,
                   label=f"{'without' if name == 'base' else 'with'} LoRA")
    ax[0].set_xlabel("total probability on the option codes"); ax[0].set_title("where the next token goes"); ax[0].legend()
    cmap = {lab: plt.cm.tab20(j) for j, lab in enumerate(I4)}
    for j, name in enumerate(["base", "lora"]):
        z = Z[name]; a_ = ax[j + 1]
        a_.scatter(z[:, 0], z[:, 1], s=8, c=[cmap[v] for v in y])
        a_.set_title(f"state at the answer position, {'without' if name == 'base' else 'with'} LoRA\n"
                     f"color = correct intent (linear probe {s6.loc[name, 'probe_accuracy_intent']:.2f})")
        a_.set_xticks([]); a_.set_yticks([])
    savefig("exp6_xray")

In [ ]:
# 10. Package the results
zpath = ROOT / f"lev_bench_{RUN_ID}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(ROOT))
log.info(f"done: {zpath}")